# Interval gap analysis




## Setup


In [ ]:
from pathlib import Path
from math import sqrt

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in (start, *start.parents):
        if (p / 'pyproject.toml').exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())

RESULTS_ROOT = REPO_ROOT / 'results'
RUN_TAG = '2026-05-26_08-56-22'
RUN_PREFIX = f'synthetic_{RUN_TAG}_oscillatory_sequential_branching'

SEEDS = [42, 43, 44, 45, 46]
STRATEGY_DIRS = {
    'active': 'active_w2_matern5_2_variance',
    'random': 'random_w2_matern5_2',
    'uniform': 'uniform_w2_matern5_2',
}

INTERVAL_SUFFIXES = {
    # Individual interval size 0.20: event1=[0.15, 0.35], event2=[0.55, 0.75].
    'interval_width_0p20': 'e1_015_035_e2_055_075',
    # Individual interval size 0.10: event1=[0.25, 0.35], event2=[0.65, 0.75].
    'interval_width_0p10': 'e1_025_035_e2_065_075',
    # Individual interval size 0.05: default event1=[0.30, 0.35], event2=[0.70, 0.75].
    'interval_width_0p05_default': 'e1_030_035_e2_070_075',
}



def run_spec(strategy: str, seed: int, suffix: str) -> Path:
    strategy_dir = STRATEGY_DIRS[strategy]
    return RESULTS_ROOT / f'{RUN_PREFIX}_{strategy_dir}_seed{seed}_{suffix}'


def runs_for_interval(suffix: str) -> list[Path]:
    return [
        run_spec(strategy, seed, suffix)
        for seed in SEEDS
        for strategy in STRATEGY_DIRS
    ]


TABLE_RUN_GROUPS: dict[str, list[Path]] = {
    label: runs_for_interval(suffix)
    for label, suffix in INTERVAL_SUFFIXES.items()
}

assert sum(len(group) for group in TABLE_RUN_GROUPS.values()) == 45

TABLE_RUN_DIRS = [
    run_dir
    for group in TABLE_RUN_GROUPS.values()
    for run_dir in group
]

missing_run_dirs = [run_dir for run_dir in TABLE_RUN_DIRS if not run_dir.exists()]
if missing_run_dirs:
    raise FileNotFoundError(
        'Missing interval sweep run directories:\n' + '\n'.join(map(str, missing_run_dirs))
    )




## Load Metrics


In [ ]:
def read_event_windows(cfg_text: str):
    lines = cfg_text.splitlines()

    def read_event(name: str):
        vals = []
        for i, line in enumerate(lines):
            if line.strip() == f"{name}:":
                for j in range(i + 1, min(i + 3, len(lines))):
                    l = lines[j].strip()
                    if l.startswith('- '):
                        try:
                            vals.append(float(l[2:]))
                        except Exception:
                            pass
                break
        return vals

    return read_event('event1'), read_event('event2')


def interval_label(event1, event2) -> str:
    return f"e1_{event1[0]:.2f}_{event1[1]:.2f}_e2_{event2[0]:.2f}_{event2[1]:.2f}"


def collect_step_metrics(run_dirs: list[Path]) -> pd.DataFrame:
    rows = []
    missing = []
    for run_dir in run_dirs:
        metrics_path = run_dir / 'metrics_by_step.csv'
        cfg_path = run_dir / '.hydra' / 'config.yaml'
        if not metrics_path.exists() or not cfg_path.exists():
            missing.append(str(run_dir))
            continue

        cfg_text = cfg_path.read_text()
        event1, event2 = read_event_windows(cfg_text)
        if len(event1) != 2 or len(event2) != 2:
            continue

        seed_val = None
        for line in cfg_text.splitlines():
            if line.strip().startswith('seed:'):
                try:
                    seed_val = int(line.split(':', 1)[1].strip())
                except Exception:
                    seed_val = None
                break

        df = pd.read_csv(metrics_path)
        if df.empty:
            continue

        df['strategy_base'] = df['strategy'].str.split(':').str[0]
        df = df[df['strategy_base'].isin(['active', 'uniform', 'random'])]
        if df.empty:
            continue

        df['event1_start'] = event1[0]
        df['event1_end'] = event1[1]
        df['event2_start'] = event2[0]
        df['event2_end'] = event2[1]
        df['event1_width'] = event1[1] - event1[0]
        df['event2_width'] = event2[1] - event2[0]
        df['total_width'] = df['event1_width'] + df['event2_width']
        df['individual_interval_width'] = 0.5 * df['total_width']
        df['interval'] = interval_label(event1, event2)
        df['interval_size'] = df['individual_interval_width']
        df['seed'] = seed_val
        df['run_dir'] = str(run_dir)

        rows.append(df)

    if missing:
        raise FileNotFoundError('Missing metrics/config for:\n' + '\n'.join(missing))
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


def load_interval_metrics(
    run_dirs: list[Path] = TABLE_RUN_DIRS,
) -> pd.DataFrame:
    df = collect_step_metrics(run_dirs)
    if df.empty:
        return df

    steps_by_interval = df.groupby('interval')['step'].unique()
    common_steps = sorted(set(steps_by_interval.iloc[0]))
    for steps in steps_by_interval.iloc[1:]:
        common_steps = sorted(set(common_steps).intersection(set(steps)))
    return df[df['step'].isin(common_steps)].copy()


df = load_interval_metrics()




## LaTeX Table


In [ ]:
def mean_ci90(series: pd.Series):
    # 90% CI using normal approximation: mean +/- 1.645 * SEM.
    series = series.dropna()
    n = series.shape[0]
    if n == 0:
        return float('nan'), float('nan')
    mean = series.mean()
    sem = series.std(ddof=1) / sqrt(n) if n > 1 else 0.0
    return mean, 1.645 * sem


def interval_size_from_label(label: str) -> float:
    parts = label.split('_')
    if len(parts) < 6:
        raise ValueError(f'Unexpected interval label: {label}')
    a = float(parts[1])
    b = float(parts[2])
    c = float(parts[4])
    d = float(parts[5])
    return 0.5 * ((b - a) + (d - c))


def per_seed_step_improvement(df: pd.DataFrame, metric_col: str) -> pd.DataFrame:
    pivot = df.pivot_table(
        index=['interval', 'seed', 'step'],
        columns='strategy_base',
        values=metric_col,
        aggfunc='mean',
    ).reset_index()
    pivot = pivot.dropna(subset=['active', 'uniform', 'random'], how='any')
    if pivot.empty:
        return pd.DataFrame()
    pivot['imp_vs_uniform'] = (pivot['uniform'] - pivot['active']) / pivot['uniform']
    pivot['imp_vs_random'] = (pivot['random'] - pivot['active']) / pivot['random']
    return (
        pivot.groupby(['interval', 'seed'])
        .agg(
            imp_vs_uniform=('imp_vs_uniform', 'mean'),
            imp_vs_random=('imp_vs_random', 'mean'),
        )
        .reset_index()
    )


def build_latex_table(df: pd.DataFrame) -> str:
    if df.empty:
        print('No data for LaTeX table')
        return ''

    per_seed_uniform = per_seed_step_improvement(df, 'uniform_metric')
    per_seed_velocity = per_seed_step_improvement(df, 'velocity_metric')

    rows = []
    intervals = sorted(df['interval'].unique(), key=interval_size_from_label)
    for interval in intervals:
        size = interval_size_from_label(interval)
        u = per_seed_uniform[per_seed_uniform['interval'] == interval]
        v = per_seed_velocity[per_seed_velocity['interval'] == interval]

        m_ru, ci_ru = mean_ci90(u['imp_vs_random'])
        m_uu, ci_uu = mean_ci90(u['imp_vs_uniform'])
        m_rv, ci_rv = mean_ci90(v['imp_vs_random'])
        m_uv, ci_uv = mean_ci90(v['imp_vs_uniform'])

        rows.append({
            'size': size,
            'AR_MWE': (m_ru, ci_ru),
            'AU_MWE': (m_uu, ci_uu),
            'AR_wMWE': (m_rv, ci_rv),
            'AU_wMWE': (m_uv, ci_uv),
        })

    def fmt_pm(pair) -> str:
        mean, ci = pair
        if np.isnan(mean) or np.isnan(ci):
            return '---'
        return f'{mean:.3f} \\pm {ci:.3f}'

    lines = [
        r'\begin{table}[h]',
        r'\centering',
        r'\caption{Average relative improvement (mean $\pm$ 90\% CI) across seeds. MWE uses uniform\_metric, w-MWE uses velocity\_metric.}',
        r'\label{tab:interval_improvement}',
        r'\begin{tabular}{l cc cc}',
        r'\toprule',
        r' & \multicolumn{2}{c}{Active vs Uniform} & \multicolumn{2}{c}{Active vs Random} \\',
        r'Interval length & Av. imp. MWE & Av. imp. w-MWE & Av. imp. MWE & Av. imp. w-MWE \\',
        r'\midrule',
    ]
    for row in rows:
        lines.append(
            f"{row['size']:.2f} & {fmt_pm(row['AU_MWE'])} & {fmt_pm(row['AU_wMWE'])} & "
            f"{fmt_pm(row['AR_MWE'])} & {fmt_pm(row['AR_wMWE'])} \\\\"
        )
    lines.extend([
        r'\bottomrule',
        r'\end{tabular}',
        r'\end{table}',
    ])
    latex = '\n'.join(lines)
    print(latex)
    return latex


latex_table = build_latex_table(df)


\begin{table}[h]
\centering
\caption{Average relative improvement (mean $\pm$ 90\% CI) across seeds. MWE uses uniform\_metric, w-MWE uses velocity\_metric.}
\label{tab:interval_improvement}
\begin{tabular}{l cc cc}
\toprule
 & \multicolumn{2}{c}{Active vs Uniform} & \multicolumn{2}{c}{Active vs Random} \\
Interval length & Av. imp. MWE & Av. imp. w-MWE & Av. imp. MWE & Av. imp. w-MWE \\
\midrule
0.05 & 0.231 \pm 0.009 & 0.357 \pm 0.011 & 0.342 \pm 0.122 & 0.454 \pm 0.094 \\
0.10 & 0.189 \pm 0.005 & 0.277 \pm 0.004 & 0.397 \pm 0.104 & 0.464 \pm 0.077 \\
0.20 & -0.172 \pm 0.011 & 0.071 \pm 0.005 & 0.118 \pm 0.080 & 0.295 \pm 0.057 \\
\bottomrule
\end{tabular}
\end{table}
